In [18]:
#Pacotes necessários

import getdist
from getdist import mcsamples, plots
import matplotlib.pyplot as plt
import os
os.chdir(r"C:\Users\Paula\Projeto_LateDE\LCDM_Omegak\Resultados_MCMC")  #Indica o diretório onde estão os arquivos .txt que serão necessários



In [19]:
#Carrega e nomeia as cadeias MCMC para facilitar quando for chamar os dados

chain1 = mcsamples.loadMCSamples("chains/MCMC_DESI")
chain2 = mcsamples.loadMCSamples("chains/MCMC_CMB")
chain3 = mcsamples.loadMCSamples("chains/MCMC_DESI_CMB")

for ch in [chain1, chain2, chain3]:
    ch.removeBurn(0.3)   #Remove 30% das amostras iniciais (burn-in), sugestão de boas práticas para análise de cadeias MCMC
    


In [6]:
#Calcula e imprime os valores de Ω_m com incertezas: Parte 1

def omega_m_stats(chain):
    stats = chain.getParams()
    samples = stats.omegam   
    mean = samples.mean()
    std  = samples.std()
    return mean, std


In [8]:
#Calcula e imprime os valores de Ω_m com incertezas: Parte 2

results = {
    "MCMC_DESI": omega_m_stats(chain1),
    "MCMC_CMB":    omega_m_stats(chain2),
    "MCMC_DESI_CMB":     omega_m_stats(chain3),
}

for key, (mean, std) in results.items():
    print(f"{key}: Ωm = {mean:.4f} ± {std:.4f}")


MCMC_DESI: Ωm = 0.2985 ± 0.0106
MCMC_CMB: Ωm = 0.4824 ± 0.0625
MCMC_DESI_CMB: Ωm = 0.3029 ± 0.0033


In [ ]:
#Calcula e imprime os valores de H_0 e 10^3 Ω_k com incertezas

# =========================
# Função para preparar chain
# =========================

def prepare_chain(chain):
    chain.deleteFixedParams()

    if 'omegak1000' not in chain.getParamNames().list():
        omegak_1000 = chain.getParams().omegak * 1000
        chain.addDerived(
            omegak_1000,
            name='omegak1000',
            label=r'10^3 \Omega_K'
        )
    return chain


# =========================
# Estatística H0
# =========================

def H0_stats(chain):
    stats = chain.getParams()
    samples = stats.H0   
    mean = samples.mean()
    std  = samples.std()
    return mean, std

# =========================
# Estatística 10^3 Omega_K
# =========================

def omegak1000_stats(chain):
    chain = prepare_chain(chain)
    stats = chain.getParams()
    samples = stats.omegak1000   
    mean = samples.mean()
    std  = samples.std()
    return mean, std


# =========================
# Calcular resultados
# =========================

results_H0 = {
    "MCMC_DESI": H0_stats(chain1),
    "MCMC_CMB": H0_stats(chain2),
    "MCMC_DESI_CMB": H0_stats(chain3),
}

results_ok = {
    "MCMC_DESI": omegak1000_stats(chain1),
    "MCMC_CMB": omegak1000_stats(chain2),
    "MCMC_DESI_CMB": omegak1000_stats(chain3),
}

print("\nResultados H0\n")
for key, (mean, std) in results_H0.items():
    print(f"{key}: H0 = {mean:.4f} ± {std:.4f}")

print("\nResultados 10^3 ΩK\n")
for key, (mean, std) in results_ok.items():
    print(f"{key}: 10^3ΩK = {mean:.4f} ± {std:.4f}")


Resultados H0

MCMC_DESI: H0 = 72.4406 ± 6.6525
MCMC_CMB: H0 = 54.4615 ± 3.6121
MCMC_DESI_CMB: H0 = 68.6686 ± 0.3231

Resultados 10^3 ΩK

MCMC_DESI: 10^3ΩK = -3.3881 ± 33.9483
MCMC_CMB: 10^3ΩK = -43.8412 ± 17.0653
MCMC_DESI_CMB: 10^3ΩK = 2.9256 ± 1.3488


In [ ]:
#Calcula e imprime os valores de H_0 e 10^3 Ω_k com incertezas
# Calcula a média e erro assimétrico, quando existente, e entrega o resultado em formato LaTeX

def mean_sigma(chain, parname):
    samples = getattr(chain.getParams(), parname)
    return samples.mean(), samples.std(), chain.getInlineLatex(parname, limit=1)

params = ['H0', 'omegam', 'omegak1000']

for name, chain in [
    ("MCMC_DESI", chain1),
    ("MCMC_CMB", chain2),
    ("MCMC_DESI_CMB", chain3),
   
]:
    print(f"\n{name}")
    for p in params:
        mean, std, latex = mean_sigma(chain, p)
        #print(f"{p} = {mean:.3f} ± {std:.3f}")
        print(latex) 



MCMC_DESI
H_0 = 72^{+9}_{-6}
\Omega_\mathrm{m} = 0.298\pm 0.011
10^3 \Omega_K = 1\pm 34

MCMC_CMB
H_0 = 54.5^{+3.3}_{-3.7}
\Omega_\mathrm{m} = 0.483^{+0.052}_{-0.071}
10^3 \Omega_K = -44^{+19}_{-14}

MCMC_DESI_CMB
H_0 = 68.67\pm 0.33
\Omega_\mathrm{m} = 0.3029\pm 0.0034
10^3 \Omega_K = 2.9\pm 1.4
